<a href="https://colab.research.google.com/github/krupa-16/BIS_LAB/blob/main/1BM24CS140_CloudcomputingResourceAllocation_GreyWolfOptimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import random

# Tasks: CPU, Memory, Workload
tasks = [
    {"cpu": 2, "mem": 2, "work": 10},
    {"cpu": 1, "mem": 1, "work": 6},
    {"cpu": 3, "mem": 2, "work": 12},
    {"cpu": 2, "mem": 3, "work": 8}
]

# Virtual Machines: CPU, Memory, Cost
vms = [
    {"cpu": 4, "mem": 6, "rate": 2.0},
    {"cpu": 6, "mem": 8, "rate": 3.0},
    {"cpu": 3, "mem": 4, "rate": 1.5}
]

# Convert continuous positions into VM assignments
def decode(position):
    return [
        max(0, min(2, int(round(x))))
        for x in position
    ]

# Calculate fitness
def fitness(position):
    allocation = decode(position)

    cpu = [0] * 3
    mem = [0] * 3
    times = [0.0] * 3
    cost = 0

    for i, vm_id in enumerate(allocation):
        task = tasks[i]
        vm = vms[vm_id]

        cpu[vm_id] += task["cpu"]
        mem[vm_id] += task["mem"]

        execution = task["work"] / vm["cpu"]
        times[vm_id] += execution
        cost += execution * vm["rate"]

    # Penalty for exceeding resources
    penalty = 0

    for i in range(3):
        penalty += 1000 * max(0, cpu[i] - vms[i]["cpu"])
        penalty += 1000 * max(0, mem[i] - vms[i]["mem"])

    # Lower fitness is better
    return max(times) + 0.2 * cost + penalty


# GWO parameters
random.seed(42)
wolves = 30
iterations = 200
dimensions = len(tasks)

# Initialize wolf population
population = [
    [random.uniform(0, 2) for _ in range(dimensions)]
    for _ in range(wolves)
]

best_solution = None

# Main GWO loop
for iteration in range(iterations):

    population.sort(key=fitness)

    alpha = population[0].copy()
    beta = population[1].copy()
    delta = population[2].copy()

    if best_solution is None or fitness(alpha) < fitness(best_solution):
        best_solution = alpha.copy()

    a = 2 - 2 * iteration / (iterations - 1)

    new_population = []

    for wolf in population:
        new_position = []

        for j in range(dimensions):

            positions = []

            for leader in [alpha, beta, delta]:
                r1 = random.random()
                r2 = random.random()

                A = 2 * a * r1 - a
                C = 2 * r2

                D = abs(C * leader[j] - wolf[j])
                positions.append(leader[j] - A * D)

            new_value = sum(positions) / 3
            new_position.append(max(0, min(2, new_value)))

        new_population.append(new_position)

    population = new_population

# Select best solution
population.sort(key=fitness)

if fitness(population[0]) < fitness(best_solution):
    best_solution = population[0].copy()

allocation = decode(best_solution)

# Calculate final resource usage
cpu = [0] * 3
mem = [0] * 3
times = [0.0] * 3
total_cost = 0

for i, vm_id in enumerate(allocation):
    task = tasks[i]
    vm = vms[vm_id]

    cpu[vm_id] += task["cpu"]
    mem[vm_id] += task["mem"]

    execution = task["work"] / vm["cpu"]
    times[vm_id] += execution
    total_cost += execution * vm["rate"]

# Display output
print("GREY WOLF OPTIMIZER - CLOUD RESOURCE ALLOCATION")
print("------------------------------------------------")

for i, vm_id in enumerate(allocation):
    print(f"Task T{i+1} -> VM{vm_id+1}")

print("Minimum Execution Time:", round(max(times), 2))
print("Total Allocation Cost:", round(total_cost, 2))
print("Best Fitness Value:", round(fitness(best_solution), 2))
print("Resource Constraints: Satisfied")


GREY WOLF OPTIMIZER - CLOUD RESOURCE ALLOCATION
------------------------------------------------
Task T1 -> VM2
Task T2 -> VM2
Task T3 -> VM1
Task T4 -> VM3
Minimum Execution Time: 3.0
Total Allocation Cost: 18.0
Best Fitness Value: 6.6
Resource Constraints: Satisfied
